# Lab 1

# Setup

In [2]:
# Setup
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)


INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


#

In [3]:

import torch.nn as nn
from chop.nn.modules import Identity

search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16],
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],
    "linear_layer_choices": [
        nn.Linear,
        Identity,
    ],
}

# model constructor

from transformers import AutoConfig, AutoModelForSequenceClassification
from chop.tools.utils import deepsetattr

def construct_model(trial):
    config = AutoConfig.from_pretrained(checkpoint)

    # Update the paramaters in the config
    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        chosen_idx = trial.suggest_int(param, 0, len(search_space[param]) - 1)
        print("chosen_idx = ", chosen_idx)
        print("search_space[param][chosen_idx]", search_space[param][chosen_idx])
        setattr(config, param, search_space[param][chosen_idx])

    trial_model = AutoModelForSequenceClassification.from_config(config)

    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == nn.Linear:
                continue
            elif new_layer_cls == Identity:
                new_layer = Identity()
                deepsetattr(trial_model, name, new_layer)
            else:
                raise ValueError(f"Unknown layer type: {new_layer_cls}")

    return trial_model

# Compression Aware Search

In [ ]:
import copy

quantization_config = {
    "by": "type",
    "default": {
        "config": {
            "name": None,
        }
    },
    "linear": {
        "config": {
            "name": "integer",
            # data
            "data_in_width": 8,
            "data_in_frac_width": 4,
            # weight
            "weight_width": 8,
            "weight_frac_width": 4,
            # bias
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
}

pruning_config = {
    "weight": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
}

compression_config = {
    "quantize_transform_pass": quantization_config,
    "prune_transform_pass": pruning_config,
}


from chop.pipelines import CompressionPipeline
from chop import MaseGraph
import os

pipe = CompressionPipeline()


evals_1 = []
evals_2 = []
evals_3 = []

# Trainer
from chop.tools import get_trainer


def objective(trial):
    print("trial config: ", trial.params)
    # Define the model
    model = construct_model(trial)
    print("trial config: ", trial.params)
    
    # 3 
    # s in total, run trainer.train() twice
    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=5,
    )

    # Stage 1: No Compression - just training
    trainer.train()
    eval_results_1 = trainer.evaluate()
    ## Store the results
    evals_1.append((trial.params, eval_results_1["eval_accuracy"]))

    # Stage 2: Compression Aware (no Post Compression Training)
    # Move model to CPU and create MaseGraph for compression
    model.cpu()
    mg = MaseGraph(model)
    # Use a deep copy of compression_config to avoid mutation across trials
    mg, _ = pipe(mg, copy.deepcopy(compression_config))
    # Move model back to CUDA for evaluation
    model.cuda()
    eval_results_2 = trainer.evaluate()
    ## Store the results
    evals_2.append((trial.params, eval_results_2["eval_accuracy"]))

    # Stage 3: Compression Aware (with QAT)
    trainer.train()
    eval_results_3 = trainer.evaluate()
    ## Store the results
    evals_3.append((trial.params, eval_results_3["eval_accuracy"]))

    # Set the model as an attribute so we can fetch it later
    trial.set_user_attr("model", model)

    return eval_results_3["eval_accuracy"]


from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
)

study.optimize(
    objective,
    n_trials=20,
    timeout=60 * 60 * 24,
)

[I 2026-02-02 23:28:05,807] A new study created in memory with name: bert-tiny-nas-study


trial config:  {}
chosen_idx =  1
search_space[param][chosen_idx] 4
chosen_idx =  2
search_space[param][chosen_idx] 8
chosen_idx =  0
search_space[param][chosen_idx] 128
chosen_idx =  4
search_space[param][chosen_idx] 2048


/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.modules.identity.Identity'> which is of type type.
  warnings.warn(message)


trial config:  {'num_layers': 1, 'num_heads': 2, 'hidden_size': 0, 'intermediate_size': 4, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}


/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.
INFO     Getting dummy input for prajjwal1/bert-tiny.


tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
tensor([[ 101, 9932, 2089, 2202, 2058, 1996, 2088, 2028, 2154,  102],
        [ 101, 2023, 2003, 2339, 2017, 2323, 4553, 4748, 4877,  102]])
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])
tensor([[[[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]],


        [[[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]]]])
tensor([[[[1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
          [1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
   

INFO     Pruning module: bert_encoder_layer_0_attention_self_query
INFO     Pruning module: bert_encoder_layer_0_attention_self_value
INFO     Pruning module: bert_encoder_layer_0_attention_output_dense
INFO     Pruning module: bert_encoder_layer_0_intermediate_dense
INFO     Pruning module: bert_encoder_layer_0_output_dense
INFO     Pruning module: bert_encoder_layer_1_attention_self_value
INFO     Pruning module: bert_encoder_layer_1_intermediate_dense
INFO     Pruning module: bert_encoder_layer_1_output_dense
INFO     Pruning module: classifier


Step,Training Loss
500,0.336900
1000,0.320000
1500,0.347400
2000,0.337400
2500,0.315200
3000,0.339900


[I 2026-02-02 23:33:21,803] Trial 1 finished with value: 0.87392 and parameters: {'num_layers': 0, 'num_heads': 3, 'hidden_size': 2, 'intermediate_size': 3, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}. Best is trial 1 with va

The final model will be given by 

```py
model = study.best_trial.user_attrs["model"].cpu()
mg = MaseGraph(model)
mg, _ = pipe(mg, compression_config)
```


# Lab 1

In [ ]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from pathlib import Path
from chop import MaseGraph
from chop.tools import get_tokenized_dataset, get_trainer
import chop.passes as passes
import json

# setting data_in_width
widths = [i for i in range(4, 33)]
print(f"Widths to be evaluated: {widths}")

# dictionary for storing eval results 
eval_dict = {}

"""Loop starts"""
for width in widths:
    # width = 4

    # Load starting model
    mg = MaseGraph.from_checkpoint(f"{Path.home()}/tutorial_2_lora")

    # Load dataset, tokenizer
    dataset, tokenizer = get_tokenized_dataset(
        dataset=dataset_name,
        checkpoint=tokenizer_checkpoint,
        return_tokenizer=True,
    )


    # PTQ
    quantization_config = {
        "by": "type",
        "default": {
            "config": {
                "name": None,
            }
        },
        "linear": {
            "config": {
                "name": "integer",
                # data
                "data_in_width": 8,
                "data_in_frac_width": 4,
                # weight
                "weight_width": 8,
                "weight_frac_width": 4,
                # bias
                "bias_width": 8,
                "bias_frac_width": 4,
            }
        },
    }
    # Overwrite these with numbers from 4 to 32
    quantization_config["linear"]["config"]["data_in_width"] =  width
    quantization_config["linear"]["config"]["weight_width"] =   width
    quantization_config["linear"]["config"]["bias_width"] =     width

    mg, _ = passes.quantize_transform_pass(
        mg,
        pass_args=quantization_config,
    )

    ## Evaluate PTQ
    trainer = get_trainer(
        model=mg.model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
    )
    eval_results = trainer.evaluate()
    print(f"PTQ: Evaluation accuracy for width {width}: {eval_results['eval_accuracy']}")
    mg.export(f"{Path.home()}/tutorial_3_ptq_width_{width}")

    # QAT
    trainer.train()
    eval_results = trainer.evaluate()
    print(f"QAT: Evaluation accuracy for width {width}: {eval_results['eval_accuracy']}")
    mg.export(f"{Path.home()}/tutorial_4_qat_width_{width}")

    
    # Save eval results
    eval_dict[width] = eval_results['eval_accuracy']
    json.dump(eval_dict, open("task_3_eval_dict.json", "w"))

/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Widths to be evaluated: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


INFO     Exporting MaseGraph to /home/ism/tutorial_3_ptq_width_4.pt, /home/ism/tutorial_3_ptq_width_4.mz
INFO     Exporting GraphModule to /home/ism/tutorial_3_ptq_width_4.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /home/ism/tutorial_3_ptq_width_4.mz


PTQ: Evaluation accuracy for width 4: 0.66356


Step,Training Loss
500,0.620200
1000,0.543700
1500,0.499600
2000,0.476500
2500,0.476500
3000,0.473200


INFO     Exporting MaseGraph to /home/ism/tutorial_4_qat_width_4.pt, /home/ism/tutorial_4_qat_width_4.mz
INFO     Exporting GraphModule to /home/ism/tutorial_4_qat_width_4.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /home/ism/tutorial_4_qat_width_4.mz


QAT: Evaluation accuracy for width 4: 0.79256


WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.
/home/ism/ADL/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


KeyboardInterrupt: 